## Voice Features - Linear Mixed Model (Baseline & Interaction effects)

### Imports

In [ ]:
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import shapiro, kstest

import seaborn as sns
from pathlib import Path
from eda import EDAPlots as eda  
from pathlib import Path

### Data

In [ ]:
# Dataset
DATA_PATH = ''
df = pd.read_excel(DATA_PATH)

# Column names
NR_COL     = "Nr" 
TIME_COL   = "Timepoint" 
TASK_COL   = "Task"
GROUP_COL  = "Group"         
AGE_COL    = "Age"
SEX_COL    = "Sex"            
EQUIP_COL  = "Voice_Equipment" 

# Voice features 
voice_features = [
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "F0semitoneFrom27.5Hz_sma3nz_percentile20.0",
    "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",
    "F0semitoneFrom27.5Hz_sma3nz_percentile80.0",
    "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope",
    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "loudness_sma3_percentile20.0",
    "loudness_sma3_percentile50.0",
    "loudness_sma3_percentile80.0",
    "loudness_sma3_pctlrange0-2",
    "loudness_sma3_meanRisingSlope",
    "loudness_sma3_stddevRisingSlope",
    "loudness_sma3_meanFallingSlope",
    "loudness_sma3_stddevFallingSlope",
    "spectralFlux_sma3_amean",
    "spectralFlux_sma3_stddevNorm",
    "mfcc1_sma3_amean",
    "mfcc1_sma3_stddevNorm",
    "mfcc2_sma3_amean",
    "mfcc2_sma3_stddevNorm",
    "mfcc3_sma3_amean",
    "mfcc3_sma3_stddevNorm",
    "mfcc4_sma3_amean",
    "mfcc4_sma3_stddevNorm",
    "jitterLocal_sma3nz_amean",
    "jitterLocal_sma3nz_stddevNorm",
    "shimmerLocaldB_sma3nz_amean",
    "shimmerLocaldB_sma3nz_stddevNorm",
    "HNRdBACF_sma3nz_amean",
    "HNRdBACF_sma3nz_stddevNorm",
    "logRelF0-H1-H2_sma3nz_amean",
    "logRelF0-H1-H2_sma3nz_stddevNorm",
    "logRelF0-H1-A3_sma3nz_amean",
    "logRelF0-H1-A3_sma3nz_stddevNorm",
    "F1frequency_sma3nz_amean",
    "F1frequency_sma3nz_stddevNorm",
    "F1bandwidth_sma3nz_amean",
    "F1bandwidth_sma3nz_stddevNorm",
    "F1amplitudeLogRelF0_sma3nz_amean",
    "F1amplitudeLogRelF0_sma3nz_stddevNorm",
    "F2frequency_sma3nz_amean",
    "F2frequency_sma3nz_stddevNorm",
    "F2bandwidth_sma3nz_amean",
    "F2bandwidth_sma3nz_stddevNorm",
    "F2amplitudeLogRelF0_sma3nz_amean",
    "F2amplitudeLogRelF0_sma3nz_stddevNorm",
    "F3frequency_sma3nz_amean",
    "F3frequency_sma3nz_stddevNorm",
    "F3bandwidth_sma3nz_amean",
    "F3bandwidth_sma3nz_stddevNorm",
    "F3amplitudeLogRelF0_sma3nz_amean",
    "F3amplitudeLogRelF0_sma3nz_stddevNorm",
    "alphaRatioV_sma3nz_amean",
    "alphaRatioV_sma3nz_stddevNorm",
    "hammarbergIndexV_sma3nz_amean",
    "hammarbergIndexV_sma3nz_stddevNorm",
    "slopeV0-500_sma3nz_amean",
    "slopeV0-500_sma3nz_stddevNorm",
    "slopeV500-1500_sma3nz_amean",
    "slopeV500-1500_sma3nz_stddevNorm",
    "spectralFluxV_sma3nz_amean",
    "spectralFluxV_sma3nz_stddevNorm",
    "mfcc1V_sma3nz_amean",
    "mfcc1V_sma3nz_stddevNorm",
    "mfcc2V_sma3nz_amean",
    "mfcc2V_sma3nz_stddevNorm",
    "mfcc3V_sma3nz_amean",
    "mfcc3V_sma3nz_stddevNorm",
    "mfcc4V_sma3nz_amean",
    "mfcc4V_sma3nz_stddevNorm",
    "alphaRatioUV_sma3nz_amean",
    "hammarbergIndexUV_sma3nz_amean",
    "slopeUV0-500_sma3nz_amean",
    "slopeUV500-1500_sma3nz_amean",
    "spectralFluxUV_sma3nz_amean",
    "loudnessPeaksPerSec",
    "VoicedSegmentsPerSec",
    "MeanVoicedSegmentLengthSec",
    "StddevVoicedSegmentLengthSec",
    "MeanUnvoicedSegmentLength",
    "StddevUnvoicedSegmentLength",
    "equivalentSoundLevel_dBp",
]

# Sensitive subset
sensitive_features = [
    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "loudness_sma3_percentile20.0",
    "loudness_sma3_percentile50.0",
    "loudness_sma3_percentile80.0",
    "loudness_sma3_pctlrange0-2",
    "loudness_sma3_meanRisingSlope",
    "loudness_sma3_stddevRisingSlope",
    "loudness_sma3_meanFallingSlope",
    "loudness_sma3_stddevFallingSlope",
    "shimmerLocaldB_sma3nz_amean",
    "shimmerLocaldB_sma3nz_stddevNorm",
    "HNRdBACF_sma3nz_amean",
    "HNRdBACF_sma3nz_stddevNorm",
    "equivalentSoundLevel_dBp",
]

EQUIPMENT_FOR_LOUDNESS = 2

# Filter for equipment-sensitive features
def filter_for_features(df, feature_list, equip_sensitive, equip_required):
    if any(f in equip_sensitive for f in feature_list):
        return df[df[EQUIP_COL] == equip_required].copy()
    else:
        return df.copy()

# Convert categorical variables to category dtype
categorical_vars = [GROUP_COL, TIME_COL, TASK_COL, SEX_COL, EQUIP_COL]

for col in categorical_vars:
    if col in df.columns:
        df[col] = df[col].astype("category")

# Convert identifier variable to string/object
df[NR_COL] = df[NR_COL].astype(str)

# Sanity check summary
print("\nVariable type summary:")
print(df.dtypes.loc[[NR_COL] + categorical_vars].to_frame("dtype"))
print("\nUnique categories per categorical variable:")
for col in categorical_vars:
    print(f"{col}: {list(df[col].cat.categories)}")


### Preprocessing & Data preparation

#### Preprocessing (filtering, exclusion, QC & standardisation)

In [ ]:
def preprocess_features_rq3(
    df: pd.DataFrame,
    feature_cols: list,
    nr_col: str,
    time_col: str,
    group_col: str,
    task_col: str,                     
    equip_col: str,
    sensitive_features: list,
    equip_required: int,
    exclude_nr: list = None
):

    df_clean = df.copy()

    # --- Ensure stable categorical encoding (string-based) ---
    df_clean[GROUP_COL] = df_clean[GROUP_COL].astype(str)
    df_clean[GROUP_COL] = pd.Categorical(df_clean[GROUP_COL], categories=["2", "1"], ordered=True) # 2 = Controls, 1 = ADHD

    df_clean[TIME_COL] = pd.Categorical(df_clean[TIME_COL].astype(str), categories=["t0", "t2"], ordered=True) # t0 = Reference
    df_clean[SEX_COL]  = pd.Categorical(df_clean[SEX_COL].astype(str), categories=["1", "2", "3"], ordered=False) # nominal control variable


    # --- Create baseline age ---
    df_clean["Age_baseline"] = (
        df_clean.loc[df_clean[TIME_COL] == "t0"].groupby(NR_COL)[AGE_COL].transform("first")
    )
    df_clean["Age_baseline"] = df_clean.groupby(NR_COL)["Age_baseline"].transform("first")
    print(df_clean[["Nr", "Timepoint", "Age", "Age_baseline"]].head(11))

    # --- 1. Exclude predefined subjects ---
    df_clean[nr_col] = df_clean[nr_col].astype(str)
    exclude_nr = [str(x) for x in exclude_nr]

    if exclude_nr:
        before = df_clean[nr_col].nunique()
        df_clean = df_clean[~df_clean[nr_col].isin(exclude_nr)].copy()
        after = df_clean[nr_col].nunique()
        print(f"Excluded {before - after} subjects: {exclude_nr}")


    qc_log = []

    # --- 2. Loop over features ---
    for feat in feature_cols:

        # --- Equipment filter for sensitive features ---
        if feat in sensitive_features:
            mask_wrong_equip = df_clean[equip_col] != equip_required
            n_wrong = df_clean.loc[mask_wrong_equip, nr_col].nunique()
            df_clean.loc[mask_wrong_equip, feat] = np.nan
            print(f"{feat}: set to NaN for {n_wrong} unique subjects (wrong equipment).")
            df_feat = df_clean[df_clean[equip_col] == equip_required].copy()
        else:
            df_feat = df_clean.copy()

        # --- 3. Outlier detection per timepoint × group × task (Tukey’s fences; non-parametric) ---
        for tp in df_feat[time_col].dropna().unique():
            df_tp = df_feat[df_feat[time_col] == tp].copy()
            if df_tp.empty:
                continue

            for grp in df_tp[group_col].dropna().unique():
                df_grp = df_tp[df_tp[group_col] == grp].copy()
                if df_grp.empty:
                    continue

                for task in df_grp[task_col].dropna().unique():
                    df_task = df_grp[df_grp[task_col] == task].copy()
                    values = df_task[feat].dropna().astype(float)
                    if values.empty:
                        continue

                    # Quartiles and IQR
                    q1, q3 = np.percentile(values, [25, 75])
                    iqr = q3 - q1

                    # Tukey fences
                    lower_std, upper_std = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                    lower_ext, upper_ext = q1 - 3.0 * iqr, q3 + 3.0 * iqr

                    # Identify outliers
                    mask_std = (values < lower_std) | (values > upper_std)
                    mask_ext = (values < lower_ext) | (values > upper_ext)

                    n_std, n_ext = mask_std.sum(), mask_ext.sum()
                    frac_std = n_std / len(values)
                    frac_ext = n_ext / len(values)

                    # --- Log all relevant info including task ---
                    qc_log.append({
                        "feature": feat,
                        "timepoint": tp,
                        "group": grp,
                        "task": task,                         
                        "n_standard_outliers": int(n_std),
                        "frac_standard_outliers": frac_std,
                        "n_extreme_outliers": int(n_ext),
                        "frac_extreme_outliers": frac_ext,
                        "iqr": iqr,
                        "q1": q1,
                        "q3": q3,
                        "lower_std_fence": lower_std,
                        "upper_std_fence": upper_std,
                        "lower_ext_fence": lower_ext,
                        "upper_ext_fence": upper_ext,
                        "equip_filtered": feat in sensitive_features,
                        "n_unique_nr": df_task[nr_col].nunique()
                    })

        # --- 4. Standardization across all timepoints ---
        values_all = df_feat[feat].dropna().astype(float)
        if not values_all.empty:
            mean_all, std_all = values_all.mean(), values_all.std(ddof=0)
            if std_all > 0:
                df_clean.loc[df_clean[feat].notna(), feat] = (
                    (df_clean.loc[df_clean[feat].notna(), feat] - mean_all) / std_all
                )
            else:
                print(f"{feat}: std = 0, skipped scaling.")

    qc_log = pd.DataFrame(qc_log)
    return df_clean, qc_log


# Define columns and parameters
EXCLUDE_NR = [] # Fill in subject ID if child needs to be exluded (i.e. pubertal voice change) 

initial_n = df[NR_COL].nunique()

df_clean, qc_log = preprocess_features_rq3(
    df=df,
    feature_cols=voice_features,
    nr_col=NR_COL,
    time_col=TIME_COL,
    group_col="Group",
    task_col=TASK_COL,                     
    equip_col=EQUIP_COL,
    sensitive_features=sensitive_features,
    equip_required=EQUIPMENT_FOR_LOUDNESS,
    exclude_nr=EXCLUDE_NR
)

qc_log = pd.DataFrame(qc_log).sort_values(by=["task", "feature", "timepoint", "group"]).reset_index(drop=True)

final_n = df_clean[NR_COL].nunique()

print(f"Initial number of subjects: {initial_n}")
print(f"Excluded subjects requested: {EXCLUDE_NR}")
print(f"Number actually excluded: {initial_n - final_n}")
print(f"Remaining subjects after exclusion: {final_n}")
print("All applicable voice features have been z-scaled (standardized).")
print(f"QC log entries: {len(qc_log)} (feature × timepoint × group combinations with outliers)")

#qc_log.to_excel("qc_log.xlsx", index=False)

#### Check completeness of timepoints

In [ ]:
# --- Check completeness of timepoints per subject ---
timepoint_check = (
    df_clean.groupby(NR_COL)[TIME_COL]
    .apply(lambda x: set(x))
    .reset_index()
    .rename(columns={TIME_COL: "available_timepoints"})
)

# Subjects missing t0 or t2
missing_t0 = timepoint_check[~timepoint_check["available_timepoints"].apply(lambda x: "t0" in x)][NR_COL].tolist()
missing_t2 = timepoint_check[~timepoint_check["available_timepoints"].apply(lambda x: "t2" in x)][NR_COL].tolist()

# Subjects with both t0 and t2
have_both = timepoint_check[timepoint_check["available_timepoints"].apply(lambda x: {"t0","t2"}.issubset(x))][NR_COL].tolist()

print(f"Subjects missing t0: {missing_t0}")
print(f"Subjects missing t2: {missing_t2}")
print(f"Total subjects with both t0 and t2: {len(have_both)}")
print(f"Subjects with both: {have_both}")


### Linear Mixed Model (Baseline + Interaction effects)

In [ ]:
# LINEAR MIXED MODEL (LMM) – Baseline Differences +  Group × Time Interaction

# ---------------- CONFIG ----------------
MIN_N_PER_GROUP = 5
OUTFILE = ".xlsx"

UNVOICED_FEATURES = [
    "alphaRatioUV_sma3nz_amean",
    "hammarbergIndexUV_sma3nz_amean",
    "slopeUV0-500_sma3nz_amean",
    "slopeUV500-1500_sma3nz_amean",
    "spectralFluxUV_sma3nz_amean",
    "StddevVoicedSegmentLengthSec",
    "MeanUnvoicedSegmentLength",
    "StddevUnvoicedSegmentLength",
]
# ----------------------------------------


# --- Variance checks ---
def _passes_variance_checks(sub: pd.DataFrame, feat: str) -> bool:
    """Skip features if too few subjects, only 1 group, or zero variance."""
    if sub.empty or feat not in sub.columns:
        return False
    if sub[GROUP_COL].nunique(dropna=True) < 2:
        return False
    for g in ("1", "2"):  # ADHD=1, HC=2
        x = pd.to_numeric(sub.loc[sub[GROUP_COL] == g, feat], errors="coerce").dropna().values
        if x.size < MIN_N_PER_GROUP:
            return False
        if len(np.unique(x)) < 2:
            return False
    return True


# --- Main analysis ---
feature_results = []

for task in df_clean[TASK_COL].unique():
    print(f"\n--- Task: {task} ---")
    df_task = df_clean[df_clean[TASK_COL] == task]

    for feat in tqdm(voice_features, desc=f"Task {task}"):

        cols_needed = [feat, GROUP_COL, TIME_COL, "Age_baseline", SEX_COL, NR_COL]
        df_feat = df_task[cols_needed].dropna().copy().reset_index(drop=True)

        # Compute n per feature *after* dropping missing values
        n_ADHD_t0 = df_feat[(df_feat[GROUP_COL] == "1") & (df_feat[TIME_COL] == "t0")][NR_COL].nunique()
        n_ADHD_t2 = df_feat[(df_feat[GROUP_COL] == "1") & (df_feat[TIME_COL] == "t2")][NR_COL].nunique()
        n_HC_t0   = df_feat[(df_feat[GROUP_COL] == "2") & (df_feat[TIME_COL] == "t0")][NR_COL].nunique()
        n_HC_t2   = df_feat[(df_feat[GROUP_COL] == "2") & (df_feat[TIME_COL] == "t2")][NR_COL].nunique()

        # --- Compute group means ---
        group_means = (
            df_feat.groupby([GROUP_COL, TIME_COL])[feat]
            .mean()
            .unstack()
        )
        mean_ADHD_t0 = mean_ADHD_t2 = mean_HC_t0 = mean_HC_t2 = np.nan
        if "1" in group_means.index:
            mean_ADHD_t0 = group_means.loc["1"].get("t0", np.nan)
            mean_ADHD_t2 = group_means.loc["1"].get("t2", np.nan)
        if "2" in group_means.index:
            mean_HC_t0 = group_means.loc["2"].get("t0", np.nan)
            mean_HC_t2 = group_means.loc["2"].get("t2", np.nan)
        
        baseline_direction = (
            "ADHD > HC" if mean_ADHD_t0 > mean_HC_t0
            else "ADHD < HC" if mean_ADHD_t0 < mean_HC_t0
            else "Equal"
        )

        # --- Structural skip for UV features in phonation tasks ---
        if task.startswith("Phonation") and feat in UNVOICED_FEATURES:
            feature_results.append({
                "task": task, "feature": feat,
                "skip_reason": "structurally_invalid", "method_used": "none",
            })
            continue

        # --- Regular variance check ---
        if df_feat.empty or not _passes_variance_checks(df_feat, feat):
            feature_results.append({
                "task": task, "feature": feat,
                "skip_reason": "variance_check_failed", "method_used": "none",
            })
            continue

        df_feat["group_id"] = pd.Categorical(df_feat[NR_COL]).codes


        # --- LMM formula and terms (Controls = ref, t0 = ref) ---
        formula = (
            f'Q("{feat}") ~ C({GROUP_COL}, Treatment(reference="2")) * '
            f'C({TIME_COL}, Treatment(reference="t0")) + Age_baseline + C({SEX_COL})'
        )
        term_baseline = f'C({GROUP_COL}, Treatment(reference="2"))[T.1]'
        term_interaction = (
            f'C({GROUP_COL}, Treatment(reference="2"))[T.1]:'
            f'C({TIME_COL}, Treatment(reference="t0"))[T.t2]'
        )

        # --- LMM ---
        try:
            model = smf.mixedlm(formula, data=df_feat, groups=df_feat["group_id"], re_formula="1").fit(
                reml=True, method="powell", maxiter=1000, disp=False
            )

            # Extract baseline effect
            beta_base = model.params.get(term_baseline, np.nan)
            se_base = model.bse.get(term_baseline, np.nan)
            pval_base = model.pvalues.get(term_baseline, np.nan)
            ci_lower_base, ci_upper_base = np.nan, np.nan
            if term_baseline in model.params.index:
                ci = model.conf_int().loc[term_baseline]
                ci_lower_base, ci_upper_base = ci[0], ci[1]

            # Extract interaction effect
            beta_int = model.params.get(term_interaction, np.nan)
            se_int = model.bse.get(term_interaction, np.nan)
            pval_int = model.pvalues.get(term_interaction, np.nan)
            ci_lower_int, ci_upper_int = np.nan, np.nan
            if term_interaction in model.params.index:
                ci = model.conf_int().loc[term_interaction]
                ci_lower_int, ci_upper_int = ci[0], ci[1]
            
            # Directionality (ADHD change vs HC change)
            if pd.isna(beta_int):
                direction = "no_diff"
            elif beta_int > 0:
                direction = "ADHD ↑ vs HC"
            elif beta_int < 0:
                direction = "ADHD ↓ vs HC"
            else:
                direction = "no_diff"

            feature_results.append({
                "task": task, "feature": feat,
                "n_ADHD_t0": n_ADHD_t0, "n_ADHD_t2": n_ADHD_t2,
                "n_HC_t0": n_HC_t0, "n_HC_t2": n_HC_t2,
                "mean_ADHD_t0": mean_ADHD_t0, "mean_ADHD_t2": mean_ADHD_t2,
                "mean_HC_t0": mean_HC_t0, "mean_HC_t2": mean_HC_t2,
                "coef_baseline": beta_base, "se_baseline": se_base,
                "p_val_baseline": pval_base,
                "ci_lower_baseline": ci_lower_base, "ci_upper_baseline": ci_upper_base,
                "std_effect_baseline": beta_base / se_base if se_base and se_base > 0 else np.nan,
                "baseline_direction": baseline_direction,
                "coef_interaction": beta_int, "se_interaction": se_int,
                "p_val_interaction": pval_int,
                "ci_lower_interaction": ci_lower_int, "ci_upper_interaction": ci_upper_int,
                "std_effect": beta_int / se_int if se_int and se_int > 0 else np.nan,
                "direction_interaction": direction, 
                "df_resid": getattr(model, "df_resid", np.nan),
                "method_used": "LMM", "skip_reason": None, "convergence": str(model.converged)
            })

        except Exception as e:
            print(f"LMM failed for {task}-{feat}: {e}")
            feature_results.append({
                "task": task, "feature": feat, "skip_reason": "model_failed", "method_used": "none"
            })


# --- Collect results ---
feature_df = pd.DataFrame(feature_results)

n_models = feature_df.query("method_used == 'LMM' and skip_reason.isna()").shape[0] 
print(f"LMMs fitted: {n_models}") 

# --- FDR correction per task (separately for baseline and interaction) ---
for term_type, p_col, fdr_col in [
    ("baseline", "p_val_baseline", "fdr_p_baseline"),
    ("interaction", "p_val_interaction", "fdr_p_interaction"),
]:
    feature_df[fdr_col] = np.nan
    for task, sub in feature_df.groupby("task"):
        mask_valid = (sub["skip_reason"].isna()) & (sub[p_col].notna())
        if mask_valid.any():
            _, pvals_corr, _, _ = multipletests(sub.loc[mask_valid, p_col], method="fdr_bh")
            feature_df.loc[sub.index[mask_valid], fdr_col] = pvals_corr


# --- Significance flags ---
feature_df["p_sig_baseline"] = feature_df["p_val_baseline"] < 0.05
feature_df["q_sig_baseline"] = feature_df["fdr_p_baseline"] < 0.05
feature_df["p_sig_interaction"] = feature_df["p_val_interaction"] < 0.05
feature_df["q_sig_interaction"] = feature_df["fdr_p_interaction"] < 0.05

# --- Task summary ---
summary_list = []
for task, sub in feature_df.groupby("task"):
    # Interaction effects
    abs_eff_inter = np.abs(sub["std_effect_interaction"].dropna()) if "std_effect_interaction" in sub.columns else np.abs(sub["std_effect"].dropna())
    
    # Baseline effects
    abs_eff_base = np.abs(sub["std_effect_baseline"].dropna()) if "std_effect_baseline" in sub.columns else np.nan

    summary_list.append({
        "task": task,
        "n_p_sig_baseline": sub["p_sig_baseline"].sum(),
        "n_q_sig_baseline": sub["q_sig_baseline"].sum(),
        "mean_abs_std_effect_baseline": abs_eff_base.mean() if not isinstance(abs_eff_base, float) and not abs_eff_base.empty else np.nan,
        "max_abs_std_effect_baseline": abs_eff_base.max() if not isinstance(abs_eff_base, float) and not abs_eff_base.empty else np.nan,
        "n_p_sig_interaction": sub["p_sig_interaction"].sum(),
        "n_q_sig_interaction": sub["q_sig_interaction"].sum(),
        "mean_abs_std_effect_interaction": abs_eff_inter.mean() if not isinstance(abs_eff_inter, float) and not abs_eff_inter.empty else np.nan,
        "max_abs_std_effect_interaction": abs_eff_inter.max() if not isinstance(abs_eff_inter, float) and not abs_eff_inter.empty else np.nan
    })

summary_df = pd.DataFrame(summary_list)

# --- Save to Excel ---
with pd.ExcelWriter(OUTFILE, engine="xlsxwriter") as writer:
    feature_df.to_excel(writer, sheet_name="feature_level_results", index=False)
    summary_df.to_excel(writer, sheet_name="task_summary", index=False)

print(f"LMM feature-level analysis complete. Results saved to {OUTFILE}")


### Sensitivity Check - Ranked Data Model

In [ ]:
# LINEAR MIXED MODEL (RANKED DATA) – Only Significant Features

# Congig
MIN_N_PER_GROUP = 5
OUTFILE_RANKED_SIG = ".xlsx"

# --- Select significant features (p < .05 baseline OR interaction) ---
sig_feats = (
    feature_df.loc[
        (feature_df["p_sig_baseline"] == True) | (feature_df["p_sig_interaction"] == True),
        ["task", "feature", "q_sig_baseline", "q_sig_interaction"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Found {len(sig_feats)} task–feature pairs for ranked-data retest.")


ranked_results = []

for _, row in tqdm(sig_feats.iterrows(), total=len(sig_feats), desc="Ranked LMM (SigOnly)"):
    task, feat = row["task"], row["feature"]
    df_task = df_clean[df_clean[TASK_COL] == task]
    cols_needed = [feat, GROUP_COL, TIME_COL, "Age_baseline", SEX_COL, NR_COL]
    df_feat = df_task[cols_needed].dropna().copy().reset_index(drop=True)

    if df_feat.empty:
        continue

    # --- Rank the outcome variable ---
    df_feat[f"{feat}_ranked"] = df_feat[feat].rank(method="average")

    # --- Variance check ---
    if df_feat[f"{feat}_ranked"].nunique() < 2:
        continue

    df_feat["group_id"] = pd.Categorical(df_feat[NR_COL]).codes

    # MODEL FORMULA (Controls = ref, t0 = ref)
    formula = (
        f'Q("{feat}_ranked") ~ C({GROUP_COL}, Treatment(reference="2")) * '
        f'C({TIME_COL}, Treatment(reference="t0")) + Age_baseline + C({SEX_COL})'
    )

    term_baseline = f'C({GROUP_COL}, Treatment(reference="2"))[T.1]'
    term_interaction = (
        f'C({GROUP_COL}, Treatment(reference="2"))[T.1]:'
        f'C({TIME_COL}, Treatment(reference="t0"))[T.t2]'
    )

    try:
        model = smf.mixedlm(formula, data=df_feat, groups=df_feat["group_id"], re_formula="1").fit(
            reml=True, method="powell", maxiter=1000, disp=False
        )

        # --- Extract results ---
        beta_base = model.params.get(term_baseline, np.nan)
        se_base = model.bse.get(term_baseline, np.nan)
        pval_base = model.pvalues.get(term_baseline, np.nan)
        ci_lower_base, ci_upper_base = np.nan, np.nan
        if term_baseline in model.params.index:
            ci = model.conf_int().loc[term_baseline]
            ci_lower_base, ci_upper_base = ci[0], ci[1]

        beta_int = model.params.get(term_interaction, np.nan)
        se_int = model.bse.get(term_interaction, np.nan)
        pval_int = model.pvalues.get(term_interaction, np.nan)
        ci_lower_int, ci_upper_int = np.nan, np.nan
        if term_interaction in model.params.index:
            ci = model.conf_int().loc[term_interaction]
            ci_lower_int, ci_upper_int = ci[0], ci[1]

        # Direction (same logic as original)
        if pd.isna(beta_int):
            direction = "no_diff"
        elif beta_int > 0:
            direction = "ADHD ↑ vs HC"
        elif beta_int < 0:
            direction = "ADHD ↓ vs HC"
        else:
            direction = "no_diff"

        ranked_results.append({
            "task": task,
            "feature": feat,
            "coef_baseline": beta_base,
            "se_baseline": se_base,
            "p_val_baseline": pval_base,
            "ci_lower_baseline": ci_lower_base,
            "ci_upper_baseline": ci_upper_base,
            "std_effect_baseline": beta_base / se_base if se_base and se_base > 0 else np.nan,
            "coef_interaction": beta_int,
            "se_interaction": se_int,
            "p_val_interaction": pval_int,
            "ci_lower_interaction": ci_lower_int,
            "ci_upper_interaction": ci_upper_int,
            "std_effect": beta_int / se_int if se_int and se_int > 0 else np.nan,
            "direction_interaction": direction,
            "df_resid": getattr(model, "df_resid", np.nan),
            "convergence": str(model.converged),
        })

    except Exception as e:
        print(f"Ranked LMM failed for {task}-{feat}: {e}")
        ranked_results.append({
            "task": task,
            "feature": feat,
            "skip_reason": "model_failed",
            "method_used": "ranked_LMM",
        })

# --- Collect results ---
feature_df_ranked = pd.DataFrame(ranked_results)

# --- Significance flags ---
feature_df_ranked["p_sig_baseline"] = feature_df_ranked["p_val_baseline"] < 0.05
feature_df_ranked["p_sig_interaction"] = feature_df_ranked["p_val_interaction"] < 0.05

# --- Merge with main LMM results for comparison ---
comparison = feature_df.merge(
    feature_df_ranked,
    on=["task", "feature"],
    suffixes=("_orig", "_ranked"),
    how="inner"
)

# --- Define robustness mask (original FDR-significant AND p<.05 in ranked data) ---
comparison["robust_baseline"] = ((comparison["q_sig_baseline"] == True) & (comparison["p_val_baseline_ranked"] < 0.05))
comparison["robust_interaction"] = ((comparison["q_sig_interaction"] == True) & (comparison["p_val_interaction_ranked"] < 0.05))

comparison["q_sig_baseline_original"] = (comparison["q_sig_baseline"] == True) 
comparison["q_sig_interaction_original"] = (comparison["q_sig_interaction"] == True)


# --- Task summary: how many effects stayed significant ---
summary_list = []
for task, sub in comparison.groupby("task"):
    summary_list.append({
        "task": task,
        "n_sig_orig_baseline": sub["p_sig_baseline_orig"].sum(),
        "n_sig_ranked_baseline": sub["p_sig_baseline_ranked"].sum(),
        "frac_baseline_stable": sub["p_sig_baseline_orig"].sum() and (sub["p_sig_baseline_ranked"].sum() / sub["p_sig_baseline_orig"].sum()),
        "n_q_sig_baseline_original": sub["q_sig_baseline_original"].sum(),
        "n_robust_baseline": sub["robust_baseline"].sum(),
        "n_sig_orig_interaction": sub["p_sig_interaction_orig"].sum(),
        "n_sig_ranked_interaction": sub["p_sig_interaction_ranked"].sum(),
        "frac_interaction_stable": sub["p_sig_interaction_orig"].sum() and (sub["p_sig_interaction_ranked"].sum() / sub["p_sig_interaction_orig"].sum()),
        "n_q_sig_interaction_original": sub["q_sig_interaction_original"].sum(),
        "n_robust_interaction": sub["robust_interaction"].sum()
    })

summary_df_ranked = pd.DataFrame(summary_list)

# --- Save output ---
with pd.ExcelWriter(OUTFILE_RANKED_SIG, engine="xlsxwriter") as writer:
    comparison.to_excel(writer, sheet_name="feature_level_results", index=False)
    summary_df_ranked.to_excel(writer, sheet_name="task_summary", index=False)

print(f"\nRanked-data sensitivity check complete. Results saved to {OUTFILE_RANKED_SIG}")
display(summary_df_ranked)


### Sensitivity Check - Time Between Intervals 


In [ ]:
robust_interaction = comparison[comparison["robust_interaction"] == True][["task", "feature"]].reset_index(drop=True)

print(f"Robust interaction pairs: {len(robust_interaction)}")
display(robust_interaction.head(20))

# Create time-between-visits covariate (days) 
DATE_COL = "Date"

df_clean[DATE_COL] = pd.to_datetime(df_clean[DATE_COL], format="%d.%m.%y", errors="coerce")
df_clean[DATE_COL] = df_clean[DATE_COL].dt.normalize()
df_clean[TIME_COL] = df_clean[TIME_COL].astype(str).str.strip().str.lower()

visit_dates = (
    df_clean[[NR_COL, TIME_COL, DATE_COL]]
    .dropna()
    .groupby([NR_COL, TIME_COL], as_index=False)[DATE_COL]
    .min()
)

dates_wide = visit_dates.pivot(index=NR_COL, columns=TIME_COL, values=DATE_COL)
dates_wide["time_between_days"] = (dates_wide["t2"] - dates_wide["t0"]).dt.days

df_clean = df_clean.merge(dates_wide["time_between_days"], on=NR_COL, how="left")

print(df_clean[[NR_COL, TIME_COL, DATE_COL, "time_between_days"]].head(10))

In [ ]:
# Sensitivity LMM: robust interaction features + time_between_days 
sig_feats = robust_interaction.copy()
print(f"Running sensitivity LMM for {len(sig_feats)} robust interaction pairs.")

sensitivity_results = []

for _, row in tqdm(sig_feats.iterrows(), total=len(sig_feats), desc="Sensitivity LMM"):
    task, feat = row["task"], row["feature"]
    df_task = df_clean[df_clean[TASK_COL] == task]
    cols_needed = [feat, GROUP_COL, TIME_COL, "Age_baseline", SEX_COL, NR_COL, "time_between_days"]
    df_feat = df_task[cols_needed].dropna().copy().reset_index(drop=True)

    if df_feat.empty:
        continue

    if df_feat[feat].nunique() < 2:
        continue

    df_feat["group_id"] = pd.Categorical(df_feat[NR_COL]).codes

    formula = (
        f'Q("{feat}") ~ C({GROUP_COL}, Treatment(reference="2")) * '
        f'C({TIME_COL}, Treatment(reference="t0")) + Age_baseline + C({SEX_COL}) + time_between_days'
    )
    term_interaction = (
        f'C({GROUP_COL}, Treatment(reference="2"))[T.1]:'
        f'C({TIME_COL}, Treatment(reference="t0"))[T.t2]'
    )

    try:
        model = smf.mixedlm(formula, data=df_feat, groups=df_feat["group_id"], re_formula="1").fit(
            reml=True, method="powell", maxiter=1000, disp=False
        )

        sensitivity_results.append({
            "task": task,
            "feature": feat,
            "n_obs": len(df_feat),
            "coef_interaction_interval": model.params.get(term_interaction, np.nan),
            "p_val_interaction_interval": model.pvalues.get(term_interaction, np.nan),
            "coef_time_between": model.params.get("time_between_days", np.nan),
            "p_val_time_between": model.pvalues.get("time_between_days", np.nan),
            "convergence": str(model.converged),
        })

    except Exception as e:
        print(f"Sensitivity LMM failed for {task}-{feat}: {e}")
        sensitivity_results.append({
            "task": task,
            "feature": feat,
            "skip_reason": "model_failed",
        })

sensitivity_df = pd.DataFrame(sensitivity_results)
display(sensitivity_df)

# Save sensitivity results (time_between_days covariate) to Excel
OUTFILE_SENS = "LMM_results_Sensitivity_TimeBetween.xlsx"
sensitivity_df.to_excel(OUTFILE_SENS, index=False)
print(f"Saved sensitivity results to {OUTFILE_SENS}")

### Plots - LMM results 

#### Prepare data

In [ ]:
# CONFIGURATION

NR_COL    = "Nr"
TIME_COL  = "Timepoint"
TASK_COL  = "Task"
GROUP_COL = "Group"
SEX_COL   = "Sex"
AGE_COL   = "Age"
EQUIP_COL = "Voice_Equipment"

EXCLUDE_NR = []
EQUIPMENT_FOR_LOUDNESS = 2

# PREPROCESSING FUNCTION

def preprocess_features_plot_rq3(
    df: pd.DataFrame,
    feature_cols: list,
    nr_col: str,
    time_col: str,
    group_col: str,
    task_col: str,
    equip_col: str,
    sensitive_features: list,
    equip_required: int,
    exclude_nr: list = None,
):
    """Preprocessing for plotting (raw baseline values)."""
    df_plot = df.copy()

    # Stable categorical encoding
    df_plot[group_col] = df_plot[group_col].astype(str)
    df_plot[group_col] = pd.Categorical(df_plot[group_col].map({"1": "ADHD", "2": "Controls"}), categories=["Controls", "ADHD"], ordered=True)
    df_plot[time_col]  = pd.Categorical(df_plot[time_col].map({"t0": "Baseline", "t2": "Follow-up"}), categories=["Baseline", "Follow-up"], ordered=True)
    df_plot[SEX_COL]   = pd.Categorical(df_plot[SEX_COL].astype(str), categories=["1", "2", "3"], ordered=False)

    # Exclude subjects
    df_plot[nr_col] = df_plot[nr_col].astype(str)
    exclude_nr = [str(x) for x in exclude_nr] if exclude_nr else []
    if exclude_nr:
        before = df_plot[nr_col].nunique()
        df_plot = df_plot[~df_plot[nr_col].isin(exclude_nr)].copy()
        after = df_plot[nr_col].nunique()
        print(f"[PLOT] Excluded {before - after} subjects: {exclude_nr}")

    # Equipment filtering
    for feat in feature_cols:
        if feat in sensitive_features:
            mask_wrong = df_plot[equip_col] != equip_required
            n_wrong = df_plot.loc[mask_wrong, nr_col].nunique()
            df_plot.loc[mask_wrong, feat] = np.nan
            print(f"[PLOT] {feat}: set to NaN for {n_wrong} subjects (wrong equipment).")

    return df_plot


# Prepare data

df_plot = preprocess_features_plot_rq3(
    df=df,
    feature_cols=voice_features,
    nr_col=NR_COL,
    time_col=TIME_COL,
    group_col=GROUP_COL,
    task_col=TASK_COL,
    equip_col=EQUIP_COL,
    sensitive_features=sensitive_features,
    equip_required=EQUIPMENT_FOR_LOUDNESS,
    exclude_nr=EXCLUDE_NR,
)

# Define variable names
var_names = {
    "F0semitoneFrom27.5Hz_sma3nz_amean": "F0_Mean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm": "F0_SDnorm",
    "F0semitoneFrom27.5Hz_sma3nz_percentile20.0": "F0_p20",
    "mfcc2_sma3_stddevNorm": "MFCC2_SDnorm",
    "mfcc1V_sma3nz_stddevNorm": "MFCC1V_SDnorm",
    "StddevVoicedSegmentLengthSec": "VoicedSeg_Len_SD",
    "mfcc2_sma3_amean": "MFCC2_Mean",    
    "jitterLocal_sma3nz_amean": "Jitter_Mean",
    "F1frequency_sma3nz_stddevNorm": "F1Freq_SDnorm",
    "F1bandwidth_sma3nz_amean": "F1Band_Mean",
    "F1bandwidth_sma3nz_stddevNorm": "F1Band_SDnorm",
    "mfcc2V_sma3nz_amean": "MFCC2V_Mean",
    "alphaRatioUV_sma3nz_amean": "AlphaUV_Mean",
    "hammarbergIndexUV_sma3nz_amean": "HammUV_Mean",
    "hammarbergIndexV_sma3nz_amean": "HammV_Mean",
    "loudnessPeaksPerSec": "LoudnessPeaks_Sec",
    "F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope": "F0_MeanRisingSlope",
    "spectralFluxV_sma3nz_amean": "SpecFluxV_Mean",
    "shimmerLocaldB_sma3nz_stddevNorm": "Shimmer_SDnorm",
    "mfcc3V_sma3nz_stddevNorm": "MFCC3V_SDnorm",
    "MeanVoicedSegmentLengthSec": "VoicedSeg_Len_Mean",
    "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2": "F0_pctlRange0-2",
    "hammarbergIndexV_sma3nz_amea": "HammV_Mean",
    "F3bandwidth_sma3nz_stddevNorm": "F3Band_SDnorm",
    "slopeV500-1500_sma3nz_stddevNorm": "SlopeV500-1500_SDnorm",
    "loudness_sma3_stddevNorm": "Loudness_SDnorm",
    "F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope": "F0_MeanFallingSlope",
    "F2bandwidth_sma3nz_amean": "F2Band_Mean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope": "F0_SDnormFallingSlope",
    "shimmerLocaldB_sma3nz_amean": "Shimmer_Mean",
    "HNRdBACF_sma3nz_amean": "HNR_Mean",
    "slopeUV0-500_sma3nz_amean": "SlopeUV0-500_Mean",
    "loudness_sma3_percentile50.0": "Loudness_p50",
    "spectralFlux_sma3_stddevNorm": "SpecFlux_SDnorm",
    "spectralFluxV_sma3nz_stddevNorm": "SpecFluxV_SDnorm",
    "mfcc2_sma3_amean": "MFCC2_Mean",
    "jitterLocal_sma3nz_stddevNorm": "Jitter_SDnorm",
    "alphaRatioV_sma3nz_stddevNorm": "AlphaV_SDnorm",
    "hammarbergIndexV_sma3nz_stddevNorm": "HammV_SDnorm"
}

task_names = {
    "CTT": "Picture Description",
}

# DISPLAY
#df_plot

#### Baseline plot functions

##### Single boxplot

In [ ]:
def plot_boxplot(task, y_var, ax=None, tp="Baseline", x_var="Group", legend=True):

    # make figure if necessary
    if ax is None:
        fig, ax = plt.subplots(1)

    # get data
    data = df_plot[(df_plot.Task==task) & (df_plot.Timepoint==tp)][[y_var, x_var]]

    # boxplot
    sns.boxplot(
        data,
        x=x_var,
        y=y_var,
        fliersize=0,
        hue=x_var,
        palette={"Controls": "#B5E0A8", "ADHD": "#D5B7F5"} ,
        linewidth=2,
        linecolor="0.2",
        ax=ax
    )

    # single values
    sns.stripplot(
        data,
        x=x_var,
        y=y_var,
        hue=x_var,
        palette={"Controls": "#B5E0A8", "ADHD": "#D5B7F5"},
        size=5,
        linewidth=1,
        edgecolor="k",
        alpha=0.5,
        label="Individual values",
        ax=ax
    )

    # mean
    sns.pointplot(
        data,
        x=x_var,
        y=y_var,
        color="k",
        linewidth=1,
        alpha=1,
        markersize=8,
        zorder=1000,
        errorbar=None,
        linestyle='none',
        label="mean",
        ax=ax
    )

    # labels
    ax.annotate(task_names.get(task, task), xy=(0.5, 1.15), xycoords="axes fraction", ha="center", fontsize=12, fontweight="bold")
    ax.set_title(f"{var_names.get(y_var, y_var)}", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel(var_names.get(y_var, y_var), fontsize=10)

    # legend
    if legend:
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    else:
        ax.get_legend().remove()


# test
plot_boxplot("CTT", "F0semitoneFrom27.5Hz_sma3nz_amean", legend=False)

##### Boxplot grid

In [ ]:
def plot_boxplot_grid(task_feature_list, ncol=4, figsize=(15,10)):
    nrow = int(np.ceil(len(task_feature_list) / ncol))

    fig, axes = plt.subplots(nrow, ncol, figsize=figsize)

    for i in range(len(task_feature_list)):
        task, feat = task_feature_list[i]
        ax = axes.flatten()[i]
        plot_boxplot(task, feat, ax=ax, legend=False)

    for j in range(i+1, nrow * ncol):
        axes.flatten()[j].axis('off')
    
    handles, labels = axes.flatten()[i].get_legend_handles_labels()
    axes.flatten()[i+1].legend(handles, ["Controls", "ADHD", "Group mean"], loc="upper left")

    plt.tight_layout(h_pad=3, w_pad=3)

    return fig, axes


#### Plot FDR significant (baseline)

In [ ]:
# SELECT ROBUST BASELINE FEATURES
robust_baseline_pairs = (
    comparison.loc[comparison["robust_baseline"] == True, ["task", "feature"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"[INFO] Found {len(robust_baseline_pairs)} robust baseline task–feature pairs.")
print(robust_baseline_pairs)

# create plot
fig, axes = plot_boxplot_grid(
    task_feature_list=[(task, feat) for task, feat in robust_baseline_pairs.itertuples(index=False)], 
    ncol=4, 
    figsize=(11,7)
)

# save
fig.savefig("/Users/rbamberger/Documents/1 Studie_Promotion/2 Studie_Voice/Ergebnisse/1 Final Analysis/LMM_Voice/Plots/baseline_boxplot_fdr.pdf", bbox_inches="tight")

#### Plot nominal significant (baseline)

In [ ]:
# SELECT P-SIGNIFICANT BASELINE FEATURES (ROBUST IN ORIGINAL + RANKED)
p_sig_baseline_pairs = (
    comparison.loc[comparison["p_sig_baseline_ranked"] == True, ["task", "feature"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"[INFO] Found {len(p_sig_baseline_pairs)} p-significant (ranked) baseline task–feature pairs.")
print(p_sig_baseline_pairs)

# create plot
fig, axes = plot_boxplot_grid(
    task_feature_list=[(task, feat) for task, feat in p_sig_baseline_pairs.itertuples(index=False)], 
    ncol=6, 
    figsize=(15,20)
)

# save
fig.savefig("/Users/rbamberger/Documents/1 Studie_Promotion/2 Studie_Voice/Ergebnisse/1 Final Analysis/LMM_Voice/Plots/baseline_boxplot_nominal.pdf", bbox_inches="tight")

#### Interaction plot function

##### Single interaction plot

In [ ]:

def plot_lineplot(task, y_var, ax=None, legend=True, title_fontsize=14):

    # make figure if necessary
    if ax is None:
        fig, ax = plt.subplots(1)

    # get data
    data = df_plot[df_plot["Task"] == task][[y_var, "Timepoint", "Group", "Nr"]]

    # line plot
    sns.pointplot(
        data=data,
        x="Timepoint",
        y=y_var,
        hue="Group",
        ax=ax,
        errorbar=("ci", 95),
        dodge=0.1,
        palette={"Controls": "#B5E0A8", "ADHD": "#D5B7F5"},
        markersize=8,
        linewidth=3,
        err_kws={'linewidth': 2},
        capsize=0,
    )

    # layout
    ax.annotate(task_names.get(task, task), xy=(0.5, 1.15), xycoords="axes fraction", ha="center", fontsize=title_fontsize, fontweight="bold")
    ax.set_title(f"{var_names.get(y_var, y_var)}", fontsize=title_fontsize)
    ax.set_xlabel("")
    ax.set_ylabel(var_names.get(y_var, y_var), fontsize=10)

    # legend
    if legend:
        leg = ax.legend(loc="center left", bbox_to_anchor=(1.1, 0.5))
        leg.set_title("Mean ± 95% CI")
    else:
        ax.get_legend().remove()


plot_lineplot("CTT", "F1bandwidth_sma3nz_amean")

##### Interaction plot grid

In [ ]:
def plot_lineplot_grid(task_feature_list, ncol=2, figsize=(15,10), title_fontsize=14):
    nrow = int(np.ceil(len(task_feature_list) / ncol))

    fig, axes = plt.subplots(nrow, ncol, figsize=figsize)

    for i in range(len(task_feature_list)):
        task, feat = task_feature_list[i]
        ax = axes.flatten()[i]
        plot_lineplot(task, feat, ax=ax, legend=ax.get_subplotspec().is_last_col(), title_fontsize=title_fontsize)

    for j in range(i+1, nrow * ncol):
        axes.flatten()[j].axis('off')

    plt.tight_layout(h_pad=3, w_pad=3)

    return fig, axes


#### Plot FDR significant (interaction)

In [ ]:
robust_interaction = comparison[comparison["robust_interaction"] == True][["task", "feature"]].reset_index(drop=True)
print(f"[INFO] Found {len(robust_interaction)} robust interaction task–feature pairs.")
print(robust_interaction)

fig, axes = plot_lineplot_grid(
    task_feature_list=[(task, feat) for task, feat in robust_interaction.itertuples(index=False)], 
    ncol=2, 
    figsize=(10,3.5),
    title_fontsize=14
)

fig.savefig("/Users/rbamberger/Documents/1 Studie_Promotion/2 Studie_Voice/Ergebnisse/1 Final Analysis/LMM_Voice/Plots/interaction_lineplot_fdr.pdf", bbox_inches="tight")

#### Plot nominal significant (interaction)

In [ ]:
nominal_interaction = comparison[comparison["p_sig_interaction_ranked"] == True][["task", "feature"]].reset_index(drop=True)
print(f"[INFO] Found {len(nominal_interaction)} nominal interaction task–feature pairs.")
print(nominal_interaction)

fig, axes = plot_lineplot_grid(
    task_feature_list=[(task, feat) for task, feat in nominal_interaction.itertuples(index=False)], 
    ncol=5, 
    figsize=(20,15),
    title_fontsize=12
)

fig.savefig("/Users/rbamberger/Documents/1 Studie_Promotion/2 Studie_Voice/Ergebnisse/1 Final Analysis/LMM_Voice/Plots/interaction_lineplot_nominal.pdf", bbox_inches="tight")

### Post-Hoc Tests within ADHD group

#### Interaction Effect (t-test on change scores; ADHD only)

In [ ]:
from scipy.stats import shapiro, kstest
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, ttest_rel
from scipy.stats import t

# CONFIG
pdf_sig = "Delta_normality_interaction_ADHD_histograms.pdf"
excel_sig = "Delta_normality_interaction_ADHD_tests.xlsx"

results = []
plots = []


# HELPER FUNCTION: Preprocess t0 + t2 data for post hoc test
def preprocess_features_interaction_rq3(
    df: pd.DataFrame,
    feature_cols: list,
    nr_col: str,
    time_col: str,
    group_col: str,
    task_col: str,
    equip_col: str,
    sensitive_features: list,
    equip_required: int,
    exclude_nr: list = None,
):
    """Preprocessing for analysis."""
    df_plot = df.copy()

    # --- Stable categorical encoding ---
    df_plot[group_col] = df_plot[group_col].astype(str)
    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=["2", "1"], ordered=True)
    df_plot[time_col]  = pd.Categorical(df_plot[time_col].astype(str), categories=["t0", "t2"], ordered=True)
    df_plot["Sex"]     = pd.Categorical(df_plot["Sex"].astype(str), categories=["1", "2", "3"], ordered=False)

    # --- Exclude subjects ---
    df_plot[nr_col] = df_plot[nr_col].astype(str)
    exclude_nr = [str(x) for x in exclude_nr] if exclude_nr else []
    if exclude_nr:
        before = df_plot[nr_col].nunique()
        df_plot = df_plot[~df_plot[nr_col].isin(exclude_nr)].copy()
        after = df_plot[nr_col].nunique()
        print(f"[INTERACTION] Excluded {before - after} subjects: {exclude_nr}")

    # --- Equipment filtering for sensitive features ---
    for feat in feature_cols:
        if feat in sensitive_features:
            mask_wrong = df_plot[equip_col] != equip_required
            n_wrong = df_plot.loc[mask_wrong, nr_col].nunique()
            df_plot.loc[mask_wrong, feat] = np.nan
            print(f"[INTERACTION] {feat}: set to NaN for {n_wrong} subjects (wrong equipment).")

    print(f"[INTERACTION] Ready for post-hoc ({df_plot[nr_col].nunique()} subjects, t0 + t2).")

    return df_plot


# MAIN ANALYSIS

# --- Select p-significant ranked interaction features ---
sig_feats_int = (
    comparison.loc[comparison["p_sig_interaction_ranked"], ["task", "feature"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"[INFO] Found {len(sig_feats_int)} interaction features.")

# --- Preprocess to get raw (unscaled) values for both t0 + t2 ---
df_plot_int = preprocess_features_interaction_rq3(
    df=df,
    feature_cols=voice_features,
    nr_col=NR_COL,
    time_col=TIME_COL,
    group_col=GROUP_COL,
    task_col=TASK_COL,
    equip_col=EQUIP_COL,
    sensitive_features=sensitive_features,
    equip_required=EQUIPMENT_FOR_LOUDNESS,
    exclude_nr=EXCLUDE_NR,
)

# --- ADHD only ---
adhd_data = df_plot_int[df_plot_int[GROUP_COL] == "1"].copy()

results_int = []

# --- Loop over all significant features ---
for _, row in sig_feats_int.iterrows():
    task, feat = row["task"], row["feature"]

    df_task = adhd_data[adhd_data[TASK_COL] == task].copy()

    # Split into t0 and t2
    subj_t0 = df_task[df_task[TIME_COL] == "t0"][[NR_COL, feat]]
    subj_t2 = df_task[df_task[TIME_COL] == "t2"][[NR_COL, feat]]

    # Merge to ensure subjects with both timepoints
    df_delta = subj_t0.merge(subj_t2, on=NR_COL, suffixes=("_t0", "_t2"))
    if df_delta.empty:
        continue

    # --- Compute Δ ---
    delta = (df_delta[f"{feat}_t2"] - df_delta[f"{feat}_t0"]).dropna()
    if delta.empty or len(delta) < 3:
        continue
    
    # NORMALITY TESTS 

    sh_p = shapiro(delta)[1]
    ks_p = kstest(
        (delta - delta.mean()) / delta.std(ddof=0),
        "norm"
    )[1]
    normal_flag = (sh_p > 0.05) and (ks_p > 0.05)

    results.append({
        "task": task,
        "feature": feat,
        "n": len(delta),
        "shapiro_p": sh_p,
        "ks_p": ks_p,
        "normality_ok": normal_flag
    })

    plots.append((task, feat, delta))


# HISTOGRAM PLOTS (Δ VALUES)

with PdfPages(pdf_sig) as pdf:
    for task, feat, delta in plots:
        fig, ax = plt.subplots(figsize=(8, 4))

        max_abs = np.nanmax(np.abs(delta))
        bins = np.linspace(-max_abs, max_abs, 20)

        ax.hist(delta, bins=bins, color="gray", alpha=0.7)
        ax.axvline(0, color="red", linestyle="--", linewidth=1)
        ax.set_xlim(-max_abs, max_abs)
        ax.set_title(f"{task} | {feat}")
        ax.set_xlabel("Δ Voice Feature (t2 − t0)")
        ax.set_ylabel("Count")

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

delta_normality_df = pd.DataFrame(results)
delta_normality_df.to_excel(excel_sig, index=False)

print("Δ normality check (interaction features, ADHD only) complete.")
print(f"PDF saved to: {pdf_sig}")
print(f"Excel saved to: {excel_sig}")


# PAIRED t-TESTS 
results_ttest = []

for _, row in sig_feats_int.iterrows():
    task, feat = row["task"], row["feature"]

    df_task = adhd_data[adhd_data[TASK_COL] == task][[NR_COL, TIME_COL, feat]].dropna()
    if df_task.empty:
        continue

    # --- Split t0 / t2 ---
    subj_t0 = df_task[df_task[TIME_COL] == "t0"][[NR_COL, feat]]
    subj_t2 = df_task[df_task[TIME_COL] == "t2"][[NR_COL, feat]]

    # --- Ensure paired observations ---
    df_pair = subj_t0.merge(subj_t2, on=NR_COL, suffixes=("_t0", "_t2"))
    if df_pair.shape[0] < 3:
        continue

    t0_vals = df_pair[f"{feat}_t0"]
    t2_vals = df_pair[f"{feat}_t2"]
    delta = t2_vals - t0_vals

    # Effect size: Cohen's dz (paired design)
    dz = delta.mean() / delta.std(ddof=1) if delta.std(ddof=1) > 0 else np.nan

    # 95% CI for mean Δ
    n = len(delta)
    se = delta.std(ddof=1) / np.sqrt(n)
    ci_low, ci_high = t.interval(
        0.95,
        df=n - 1,
        loc=delta.mean(),
        scale=se
    )

 
    # Version 1: One-sample t-test on Δ vs 0
    t_delta, p_delta = ttest_1samp(delta, popmean=0, nan_policy="omit")

    # Version 2: Classical paired t-test (t2 vs t0)
    t_paired, p_paired = ttest_rel(t2_vals, t0_vals, nan_policy="omit")

    results_ttest.append({
        "task": task,
        "feature": feat,
        "n": len(delta),
        "mean_t0": t0_vals.mean(),
        "mean_t2": t2_vals.mean(),
        "mean_delta": delta.mean(),
        "std_delta": delta.std(ddof=1),
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "cohens_dz": dz,
        "t_delta_vs_0": t_delta,
        "p_delta_vs_0": p_delta,
        "t_paired_t2_t0": t_paired,
        "p_paired_t2_t0": p_paired,
    })

ttest_df = pd.DataFrame(results_ttest)
ttest_df.to_excel("Delta_posthoc_ADHD_paired_ttests.xlsx", index=False)

print("Paired t-tests (Δ vs 0 and t2 vs t0) complete.")
print("Excel saved to: Delta_posthoc_ADHD_paired_ttests.xlsx")


#### Post-hoc Linear Regression on Change Scores (ADHD only)

In [ ]:
# OLS on change scores: ADHD group only
OUTFILE = "Delta_LMM_ADHD_change_scores_26.02.26.xlsx"
conners_scales = ["hi_hyperactivity_score", "ua_inattention_score"]

# --- Select robust interaction features ---
robust_feats_int = (
    comparison.loc[comparison["robust_interaction"] == True, ["task", "feature"]]
    .drop_duplicates()
    .reset_index(drop=True)
 )
print(f"[INFO] Found {len(robust_feats_int)} robust interaction features for change-score LMM.")

# --- Use preprocessed dataset (df_clean) ---
adhd_data = df_clean[df_clean[GROUP_COL] == "1"].copy()

# --- Baseline covariates ---
baseline_cov = (
    adhd_data[adhd_data[TIME_COL] == "t0"][[NR_COL, "Age_baseline", SEX_COL]]
    .rename(columns={SEX_COL: "Sex"})
    .drop_duplicates(subset=[NR_COL])
 )

def _term_stats(model, term: str) -> dict:
    if term not in model.params.index:
        return {
            f"coef_{term}": np.nan,
            f"se_{term}": np.nan,
            f"p_{term}": np.nan,
            f"ci_lower_{term}": np.nan,
            f"ci_upper_{term}": np.nan,
            f"std_effect_{term}": np.nan,
        }
    coef = model.params.get(term, np.nan)
    se = model.bse.get(term, np.nan)
    pval = model.pvalues.get(term, np.nan)
    ci = model.conf_int().loc[term] if term in model.params.index else [np.nan, np.nan]
    std_eff = coef / se if se and se > 0 else np.nan
    return {
        f"coef_{term}": coef,
        f"se_{term}": se,
        f"p_{term}": pval,
        f"ci_lower_{term}": ci[0],
        f"ci_upper_{term}": ci[1],
        f"std_effect_{term}": std_eff,
    }

results_lmm = []

for _, row in robust_feats_int.iterrows():
    task, feat = row["task"], row["feature"]

    df_task = adhd_data[adhd_data[TASK_COL] == task].copy()
    n_ADHD_t0 = df_task[df_task[TIME_COL] == "t0"][NR_COL].nunique()
    n_ADHD_t2 = df_task[df_task[TIME_COL] == "t2"][NR_COL].nunique()
    mean_ADHD_t0 = df_task[df_task[TIME_COL] == "t0"][feat].mean()
    mean_ADHD_t2 = df_task[df_task[TIME_COL] == "t2"][feat].mean()

    # Split into t0 and t2
    subj_t0 = df_task[df_task[TIME_COL] == "t0"][[NR_COL, feat] + conners_scales]
    subj_t2 = df_task[df_task[TIME_COL] == "t2"][[NR_COL, feat] + conners_scales]

    # Merge to ensure subjects with both timepoints
    df_delta = subj_t0.merge(subj_t2, on=NR_COL, suffixes=("_t0", "_t2"))
    if df_delta.empty:
        results_lmm.append({
            "task": task,
            "feature": feat,
            "n_ADHD_t0": n_ADHD_t0,
            "n_ADHD_t2": n_ADHD_t2,
            "mean_ADHD_t0": mean_ADHD_t0,
            "mean_ADHD_t2": mean_ADHD_t2,
            "n": 0,
            "skip_reason": "no_paired_timepoints",
            "method_used": "none",
        })
        continue

    # Compute deltas
    df_delta["delta_voice"] = df_delta[f"{feat}_t2"] - df_delta[f"{feat}_t0"]
    df_delta["delta_hi"] = df_delta["hi_hyperactivity_score_t2"] - df_delta["hi_hyperactivity_score_t0"]
    df_delta["delta_ia"] = df_delta["ua_inattention_score_t2"] - df_delta["ua_inattention_score_t0"]

    df_model = (
        df_delta[[NR_COL, "delta_voice", "delta_hi", "delta_ia"]]
        .merge(baseline_cov, on=NR_COL, how="left")
        .dropna()
    )

    if df_model.shape[0] < 5 or df_model["delta_voice"].nunique() < 2:
        results_lmm.append({
            "task": task,
            "feature": feat,
            "n_ADHD_t0": n_ADHD_t0,
            "n_ADHD_t2": n_ADHD_t2,
            "mean_ADHD_t0": mean_ADHD_t0,
            "mean_ADHD_t2": mean_ADHD_t2,
            "n": len(df_model),
            "skip_reason": "insufficient_variance",
            "method_used": "none",
        })
        continue

    df_model["Sex"] = df_model["Sex"].astype(str)
    mean_delta_voice = df_model["delta_voice"].mean()

    formula = "delta_voice ~ delta_ia + delta_hi + Age_baseline + C(Sex)"

    # Fit OLS on change scores
    model = smf.ols(formula, data=df_model).fit()
    method_used = "OLS"

    row_out = {
        "task": task,
        "feature": feat,
        "n_ADHD_t0": n_ADHD_t0,
        "n_ADHD_t2": n_ADHD_t2,
        "mean_voice_t0": mean_ADHD_t0,
        "mean_voice_t2": mean_ADHD_t2,
        "mean_delta_voice": mean_delta_voice,
        "n": len(df_model),
        "df_resid": getattr(model, "df_resid", np.nan),
        "method_used": method_used,
        "skip_reason": None,
        "convergence": str(getattr(model, "converged", np.nan)),
    }

    # Terms
    for term in ["Intercept", "delta_ia", "delta_hi", "Age_baseline"]:
        row_out.update(_term_stats(model, term))

    results_lmm.append(row_out)

lmm_delta_df = pd.DataFrame(results_lmm)
lmm_delta_df.to_excel(OUTFILE, index=False)

print("Change-score LMM (ADHD only) complete.")
print(f"Excel saved to: {OUTFILE}")

### Post-Hoc Clinical Correlations (ADHD only)

##### Baseline Correlations

In [ ]:
# BASELINE CORRELATIONS (t0 features vs. baseline Conners)

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

OUTFILE = ".xlsx"
conners_scales = ["hi_hyperactivity_score", "ua_inattention_score"]

# --- Select p-significant ranked baseline features ---
sig_feats_base = comparison.loc[comparison["p_sig_baseline_ranked"], ["task", "feature"]].drop_duplicates()
print(f"[INFO] Found {len(sig_feats_base)} baseline features for correlation.")


# HELPER FUNCTION: Preprocess baseline data for correlation/plotting

def preprocess_features_plot_rq3(
    df: pd.DataFrame,
    feature_cols: list,
    nr_col: str,
    time_col: str,
    group_col: str,
    task_col: str,
    equip_col: str,
    sensitive_features: list,
    equip_required: int,
    exclude_nr: list = None,
):
    """Preprocessing for baseline correlation analysis (raw values at t0)."""
    df_plot = df.copy()

    # --- Stable categorical encoding ---
    df_plot[group_col] = df_plot[group_col].astype(str)
    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=["2", "1"], ordered=True)
    df_plot[time_col]  = pd.Categorical(df_plot[time_col].astype(str), categories=["t0", "t2"], ordered=True)
    df_plot["Sex"]     = pd.Categorical(df_plot["Sex"].astype(str), categories=["1", "2", "3"], ordered=False)

    # --- Exclude subjects ---
    df_plot[nr_col] = df_plot[nr_col].astype(str)
    exclude_nr = [str(x) for x in exclude_nr] if exclude_nr else []
    if exclude_nr:
        before = df_plot[nr_col].nunique()
        df_plot = df_plot[~df_plot[nr_col].isin(exclude_nr)].copy()
        after = df_plot[nr_col].nunique()
        print(f"[PLOT] Excluded {before - after} subjects: {exclude_nr}")

    # --- Equipment filtering for sensitive features ---
    for feat in feature_cols:
        if feat in sensitive_features:
            mask_wrong = df_plot[equip_col] != equip_required
            n_wrong = df_plot.loc[mask_wrong, nr_col].nunique()
            df_plot.loc[mask_wrong, feat] = np.nan
            print(f"[PLOT] {feat}: set to NaN for {n_wrong} subjects (wrong equipment).")

    # --- Keep only baseline rows ---
    df_plot = df_plot[df_plot[time_col] == "t0"].copy()
    print(f"[PLOT] Ready for baseline correlation ({df_plot[nr_col].nunique()} subjects).")

    return df_plot


# --- Preprocess to get original values at t0 only ---
df_plot_base = preprocess_features_plot_rq3(
    df=df,
    feature_cols=voice_features,
    nr_col=NR_COL,
    time_col=TIME_COL,
    group_col=GROUP_COL,
    task_col=TASK_COL,
    equip_col=EQUIP_COL,
    sensitive_features=sensitive_features,
    equip_required=EQUIPMENT_FOR_LOUDNESS,
    exclude_nr=EXCLUDE_NR,
)

# --- ADHD only ---
adhd_base = df_plot_base[df_plot_base[GROUP_COL] == "1"].copy()

results_base = []
for _, row in sig_feats_base.iterrows():
    task, feat = row["task"], row["feature"]
    df_task = adhd_base[adhd_base[TASK_COL] == task].copy()

    for c in conners_scales:
        df_pair = df_task[[feat, c]].dropna()
        if df_pair.empty or df_pair[feat].nunique() < 2:
            continue

        rho, pval = spearmanr(df_pair[feat], df_pair[c])

        results_base.append({
            "task": task,
            "feature": feat,
            "conners": c,
            "n": len(df_pair),
            "rho": rho,
            "p": pval,
        })

# --- FDR correction (within each Task × Conners) ---
res_base = pd.DataFrame(results_base)
res_base["q"] = np.nan
for (task, c), sub in res_base.groupby(["task", "conners"]):
    if sub["p"].notna().any():
        _, qvals, _, _ = multipletests(sub["p"], method="fdr_bh")
        res_base.loc[sub.index, "q"] = qvals

# --- Save to Excel ---
with pd.ExcelWriter(OUTFILE, engine="xlsxwriter") as writer:
    res_base.to_excel(writer, sheet_name="correlations", index=False)

print(f"[DONE] Baseline correlation analysis saved to {OUTFILE}")


##### Interaction effects Correlation

In [ ]:
# INTERACTION CORRELATIONS (Δ features vs. Δ Conners)

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests


# Config
OUTFILE = ".xlsx"
conners_scales = ["hi_hyperactivity_score", "ua_inattention_score"]
# ----------------------------------------


# HELPER FUNCTION: Preprocess t0 + t2 data for interaction correlation


def preprocess_features_interaction_rq3(
    df: pd.DataFrame,
    feature_cols: list,
    nr_col: str,
    time_col: str,
    group_col: str,
    task_col: str,
    equip_col: str,
    sensitive_features: list,
    equip_required: int,
    exclude_nr: list = None,
):
    """Preprocessing for interaction correlation analysis (Δ values)."""
    df_plot = df.copy()

    # --- Stable categorical encoding ---
    df_plot[group_col] = df_plot[group_col].astype(str)
    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=["2", "1"], ordered=True)
    df_plot[time_col]  = pd.Categorical(df_plot[time_col].astype(str), categories=["t0", "t2"], ordered=True)
    df_plot["Sex"]     = pd.Categorical(df_plot["Sex"].astype(str), categories=["1", "2", "3"], ordered=False)

    # --- Exclude subjects ---
    df_plot[nr_col] = df_plot[nr_col].astype(str)
    exclude_nr = [str(x) for x in exclude_nr] if exclude_nr else []
    if exclude_nr:
        before = df_plot[nr_col].nunique()
        df_plot = df_plot[~df_plot[nr_col].isin(exclude_nr)].copy()
        after = df_plot[nr_col].nunique()
        print(f"[INTERACTION] Excluded {before - after} subjects: {exclude_nr}")

    # --- Equipment filtering for sensitive features ---
    for feat in feature_cols:
        if feat in sensitive_features:
            mask_wrong = df_plot[equip_col] != equip_required
            n_wrong = df_plot.loc[mask_wrong, nr_col].nunique()
            df_plot.loc[mask_wrong, feat] = np.nan
            print(f"[INTERACTION] {feat}: set to NaN for {n_wrong} subjects (wrong equipment).")

    print(f"[INTERACTION] Ready for Δ correlation ({df_plot[nr_col].nunique()} subjects, t0 + t2).")

    return df_plot



# Main correlation analysis

# --- Select p-significant ranked interaction features ---
sig_feats_int = (
    comparison.loc[comparison["p_sig_interaction_ranked"], ["task", "feature"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"[INFO] Found {len(sig_feats_int)} interaction features for correlation.")

# --- Preprocess to get raw (unscaled) values for both t0 + t2 ---
df_plot_int = preprocess_features_interaction_rq3(
    df=df,
    feature_cols=voice_features,
    nr_col=NR_COL,
    time_col=TIME_COL,
    group_col=GROUP_COL,
    task_col=TASK_COL,
    equip_col=EQUIP_COL,
    sensitive_features=sensitive_features,
    equip_required=EQUIPMENT_FOR_LOUDNESS,
    exclude_nr=EXCLUDE_NR,
)

# --- ADHD only ---
adhd_data = df_plot_int[df_plot_int[GROUP_COL] == "1"].copy()

results_int = []

# --- Loop over all significant features ---
for _, row in sig_feats_int.iterrows():
    task, feat = row["task"], row["feature"]

    df_task = adhd_data[adhd_data[TASK_COL] == task].copy()

    # Split into t0 and t2
    subj_t0 = df_task[df_task[TIME_COL] == "t0"][[NR_COL, feat] + conners_scales]
    subj_t2 = df_task[df_task[TIME_COL] == "t2"][[NR_COL, feat] + conners_scales]

    # Merge to ensure subjects with both timepoints
    df_delta = subj_t0.merge(subj_t2, on=NR_COL, suffixes=("_t0", "_t2"))
    if df_delta.empty:
        continue

    # Compute Δ (t2 − t0)
    df_delta["delta_voice"] = df_delta[f"{feat}_t2"] - df_delta[f"{feat}_t0"]
    for c in conners_scales:
        df_delta[f"delta_{c}"] = df_delta[f"{c}_t2"] - df_delta[f"{c}_t0"]

    # --- Correlate ΔVoice with ΔConners ---
    for c in conners_scales:
        df_pair = df_delta[["delta_voice", f"delta_{c}"]].dropna()

        if df_pair.empty or df_pair["delta_voice"].nunique() < 2:
            continue

        rho, pval = spearmanr(df_pair["delta_voice"], df_pair[f"delta_{c}"])

        results_int.append({
            "task": task,
            "feature": feat,
            "conners": c,
            "n": len(df_pair),
            "rho": rho,
            "p": pval,
            "mean_delta_voice": df_pair["delta_voice"].mean(),
            "mean_delta_conners": df_pair[f"delta_{c}"].mean(),
        })


# FDR correction 

res_int = pd.DataFrame(results_int)
res_int["q"] = np.nan

# FDR correction within each Task × Conners combination
for (task, c), sub in res_int.groupby(["task", "conners"]):
    if sub["p"].notna().any():
        _, qvals, _, _ = multipletests(sub["p"], method="fdr_bh")
        res_int.loc[sub.index, "q"] = qvals

# Save to Excel
with pd.ExcelWriter(OUTFILE, engine="xlsxwriter") as writer:
    res_int.to_excel(writer, sheet_name="correlations", index=False)

print(f"[DONE] Interaction correlation analysis saved to {OUTFILE}")
